In [63]:
import pandas as pd
import numpy as np
import networkx as nx
from datetime import datetime
from sklearn.preprocessing import StandardScaler
from typing import List, Dict, Any, Tuple
import matplotlib.pyplot as plt
from scipy import stats
import json
from multiprocessing import Pool
import logging
import seaborn as sns
import traceback

try:
    plt.style.use('seaborn')
except:
    plt.style.use('default')

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

class CryptoOnchainAnalysis:
    def __init__(self, config: Dict[str, Any] = None):
        """Initialize the analysis class with configuration"""
        self.btc_metrics = None
        self.eth_metrics = None
        self.btc_price = None
        self.eth_price = None
        
        # Default configuration
        self.config = {
            'lag_periods': [1, 3, 5, 20],
            'num_samples': 5,
            'noise_std': 0.01,
            'significance_threshold': 0.05,
            'outlier_std': 3,
            'min_causal_strength': 0.1,
            'cache_enabled': True
        }
        
        # Update configuration if provided
        if config:
            self.config.update(config)
            
        # Initialize cache
        self._score_cache = {}
        
    def remove_outliers(self, df: pd.DataFrame, 
                       columns: List[str], 
                       n_std: int = None) -> pd.DataFrame:
        """Remove outliers from specified columns"""
        logger.info("Removing outliers...")
        if n_std is None:
            n_std = self.config['outlier_std']
            
        df_clean = df.copy()
        for col in columns:
            mean = df[col].mean()
            std = df[col].std()
            df_clean = df_clean[abs(df_clean[col] - mean) <= (n_std * std)]

        return df_clean
    
    def load_data(self, 
                btc_onchain_path: str,
                eth_onchain_path: str, 
                btc_price_path: str,
                eth_price_path: str) -> None:
        """Load and prepare all datasets"""
        try:
            print("Loading data files...")
            
            # Load onchain metrics
            self.btc_metrics = pd.read_csv(btc_onchain_path)
            self.eth_metrics = pd.read_csv(eth_onchain_path)
            
            # Load price data
            self.btc_price = pd.read_csv(btc_price_path)
            self.eth_price = pd.read_csv(eth_price_path)

            # Data cleaning - remove pi_cycle_top
            for df in [self.btc_metrics, self.eth_metrics]:
                if 'pi_cycle_top' in df.columns:
                    df.drop('pi_cycle_top', axis=1, inplace=True)       
                df.set_index('timestamp', inplace=True)
                
                for col in df.columns:
                    df[col] = df[col].astype(float)
                
                df = df.pct_change()
                df = df.dropna()

            # for df in [self.btc_price, self.eth_price]:
            #     df = df[['timestamp', 'close']]
            #     df.set_index('timestamp', inplace=True)
            #     df = df.astype(float)
                
            self.btc_price = self.btc_price[['timestamp', 'close']] 
            self.eth_price = self.eth_price[['timestamp', 'close']]
            self.btc_price.set_index('timestamp', inplace=True)
            self.eth_price.set_index('timestamp', inplace=True)
            
                
            # Set timestamp as index
            for df in [self.btc_metrics, self.eth_metrics, 
                    self.btc_price, self.eth_price]:
                df.sort_index(inplace=True)

            # Calculate returns
            for price_df in [self.btc_price, self.eth_price]:
                price_df['returns'] = np.log(
                    price_df['close'] / price_df['close'].shift(1)
                )
            
            print("Data loading completed successfully")
            
            # Print data information
            print("\nData shapes after loading:")
            print("BTC metrics shape:", self.btc_metrics.shape)
            print("ETH metrics shape:", self.eth_metrics.shape)
            print("BTC price shape:", self.btc_price.shape)
            print("ETH price shape:", self.eth_price.shape)
            print(self.btc_price.head())
            
            print("\nAvailable BTC metrics:")
            print(self.btc_metrics.columns.tolist())
            print("\nAvailable ETH metrics:")
            print(self.eth_metrics.columns.tolist())
            
            print("\nDate ranges:")
            print("BTC metrics:", self.btc_metrics.index.min(), "to", self.btc_metrics.index.max())
            print("ETH metrics:", self.eth_metrics.index.min(), "to", self.eth_metrics.index.max())
            print("BTC price:", self.btc_price.index.min(), "to", self.btc_price.index.max())
            print("ETH price:", self.eth_price.index.min(), "to", self.eth_price.index.max())
            
        except Exception as e:
            print(f"Error in data loading: {str(e)}")
            raise
            
    def _print_data_info(self):
        """Print information about loaded data"""
        print("\nData shapes after loading:")
        print("BTC metrics shape:", self.btc_metrics.shape)
        print("ETH metrics shape:", self.eth_metrics.shape)
        print("BTC price shape:", self.btc_price.shape)
        print("ETH price shape:", self.eth_price.shape)
        
        print("\nDate ranges:")
        print("BTC metrics:", self.btc_metrics.index.min(), 
              "to", self.btc_metrics.index.max())
        print("ETH metrics:", self.eth_metrics.index.min(), 
              "to", self.eth_metrics.index.max())
        print("BTC price:", self.btc_price.index.min(), 
              "to", self.btc_price.index.max())
        print("ETH price:", self.eth_price.index.min(), 
              "to", self.eth_price.index.max())
    
    def prepare_features(self) -> pd.DataFrame:
        """Prepare features with multiple lags and handle outliers"""
        try:
            logger.info("Preparing features...")
            
            # Copy and clean metrics data
            btc_data = self.btc_metrics.copy()
            eth_data = self.eth_metrics.copy()
            
            # Get column names
            btc_metrics_cols = btc_data.columns
            eth_metrics_cols = eth_data.columns
            
            # Remove outliers
            btc_data = self.remove_outliers(btc_data, btc_metrics_cols)
            eth_data = self.remove_outliers(eth_data, eth_metrics_cols)

            # Normalize metrics
            scaler = StandardScaler()
            btc_data[btc_metrics_cols] = scaler.fit_transform(btc_data[btc_metrics_cols])
            eth_data[eth_metrics_cols] = scaler.fit_transform(eth_data[eth_metrics_cols])
            
            # Merge with returns data
            btc_data = self._merge_with_returns(btc_data, self.btc_price, 'btc')
            eth_data = self._merge_with_returns(eth_data, self.eth_price, 'eth')
            
            # Create combined dataset with lagged features
            combined_data = self._create_lagged_features(btc_data, eth_data)
            
            logger.info("Feature preparation completed successfully")
            self._print_feature_info(combined_data)
            
            return combined_data
            
        except Exception as e:
            logger.error(f"Error in feature preparation: {str(e)}")
            raise
            
    def _merge_with_returns(self, metrics_data: pd.DataFrame, 
                        price_data: pd.DataFrame,
                        prefix: str) -> pd.DataFrame:
        """Merge metrics with returns data and add prefixes"""
        # Merge
        merged = metrics_data.merge(
            price_data[['returns']], 
            left_index=True, 
            right_index=True,
            how='inner'
        )
        
        # Add prefixes
        cols_to_prefix = merged.columns.drop('returns').tolist()
        merged = merged.add_prefix(f'{prefix}_')
        merged.rename(columns={f'{prefix}_returns': f'{prefix}_returns'}, 
                    inplace=True)
        
        return merged
        
    def _create_lagged_features(self, btc_data: pd.DataFrame, 
                            eth_data: pd.DataFrame) -> pd.DataFrame:
        """Create lagged features for both cryptocurrencies"""
        # Combine data
        combined_data = btc_data.merge(
            eth_data, 
            left_index=True, 
            right_index=True,
            how='inner'
        )
        
        # Get feature columns
        feature_cols = [col for col in combined_data.columns 
                    if 'returns' not in col]
        returns_cols = ['btc_returns', 'eth_returns']
        
        # Initialize lagged features
        lagged_features = {}
        
        # Create lags
        for col in feature_cols:
            for lag in self.config['lag_periods']:
                lagged_features[f'{col}_lag_{lag}'] = combined_data[col].shift(lag)
        
        # Create final dataset
        lagged_data = pd.DataFrame(lagged_features)
        lagged_data[returns_cols] = combined_data[returns_cols]
        
        # Remove missing values
        lagged_data = lagged_data.dropna()
        
        return lagged_data
        
    def _print_feature_info(self, data: pd.DataFrame):
        """Print information about prepared features"""
        print("\nPrepared dataset info:")
        print("Shape:", data.shape)
        print("Date range:", data.index.min(), "to", data.index.max())
        print("Number of features:", len(data.columns))
        print("Memory usage:", data.memory_usage().sum() / 1024**2, "MB")

    def normalize_features(self, data: pd.DataFrame) -> pd.DataFrame:
        """Normalize features using StandardScaler with error handling"""
        try:
            logger.info("Normalizing features...")
            
            # Select numeric columns
            numeric_data = data.select_dtypes(include=[np.number])
            
            # Normalize
            scaler = StandardScaler()
            normalized_data = pd.DataFrame(
                scaler.fit_transform(numeric_data),
                columns=numeric_data.columns,
                index=numeric_data.index
            )
            
            logger.info("Feature normalization completed")
            return normalized_data
            
        except Exception as e:
            logger.error(f"Error in feature normalization: {str(e)}")
            raise

    def _calculate_bic_score(self, data: pd.DataFrame, graph: nx.DiGraph) -> float:
        """Calculate BIC score for a graph with robust error handling"""
        try:
            # Check cache if enabled
            if self.config['cache_enabled']:
                cache_key = frozenset(graph.edges())
                if cache_key in self._score_cache:
                    return self._score_cache[cache_key]
            
            n_samples = len(data)
            if n_samples == 0:
                raise ValueError("Empty dataset")
                
            n_edges = len(graph.edges())
            log_likelihood = 0
            
            for node in graph.nodes():
                parents = list(graph.predecessors(node))
                if parents:
                    # Multiple regression
                    log_likelihood += self._calculate_node_likelihood(
                        data, node, parents, n_samples)
                else:
                    # Just variance
                    variance = np.var(data[node])
                    if variance <= 0:
                        return -np.inf
                    log_likelihood += -0.5 * n_samples * np.log(2 * np.pi * variance) - \
                                    0.5 * n_samples
            
            # Calculate BIC score
            bic_score = log_likelihood - 0.5 * n_edges * np.log(n_samples)
            
            # Cache result if enabled
            if self.config['cache_enabled']:
                self._score_cache[cache_key] = bic_score
                
            return bic_score
            
        except Exception as e:
            logger.error(f"Error in BIC score calculation: {str(e)}")
            return -np.inf


    def _calculate_node_likelihood(self, data: pd.DataFrame, 
                                 node: str, 
                                 parents: List[str], 
                                 n_samples: int) -> float:
        """Calculate log likelihood for a single node"""
        try:
            X = data[parents]
            y = data[node]
            
            # Add constant term
            X = np.column_stack([np.ones(len(X)), X])
            
            # Calculate coefficients using robust method
            try:
                beta = np.linalg.solve(X.T @ X + np.eye(X.shape[1]) * 1e-6, X.T @ y)
                residuals = y - X @ beta
                variance = np.var(residuals)
                
                if variance <= 0:
                    return -np.inf
                    
                return -0.5 * n_samples * np.log(2 * np.pi * variance) - \
                       0.5 * sum(residuals ** 2) / variance
                       
            except np.linalg.LinAlgError:
                return -np.inf
                
        except Exception as e:
            logger.warning(f"Error in node likelihood calculation: {str(e)}")
            return -np.inf

    def generate_causal_graph(self, data: pd.DataFrame, 
                            num_samples: int = None) -> List[nx.DiGraph]:
        """여러 번의 구조 학습을 통한 인과 그래프 생성"""
        # _generate_single_graph_with_bic 함수는 bic 값이 -inf가 아니면 무조건 그래프를 추가함.  
        # 이 또한 수정 필요 ( BIC 값이 높은애들을 추가하는 방식으로 이루어져야 하는데 BIC와 관계 없이 추가됨. )
        # _generate_single_graph_with_bic -> _apply_fges 함수로 수정 필요.
        try:
            if num_samples is None:
                num_samples = self.config['num_samples']
                
            logger.info(f"Generating {num_samples} causal graphs...")
            
            all_edges = []
            
            # 데이터 전처리는 이미 수행되어 있음
            processed_data = data
            
            for i in range(num_samples):
                # 데이터 부트스트래핑
                sample_indices = np.random.choice(
                    len(processed_data), 
                    size=len(processed_data), 
                    replace=True
                )
                sample_data = processed_data.iloc[sample_indices]
                
                # 단일 그래프 생성
                # graph = self._generate_single_graph_with_bic(sample_data)
                graph = self._generate_single_graph(sample_data, 0.01)
                all_edges.extend(graph.edges())
            
            # 엣지 빈도 계산
            edge_counts = {}
            for edge in all_edges:
                edge_counts[edge] = edge_counts.get(edge, 0) + 1
            
            # 빈도 기반 필터링
            threshold = num_samples * 0.2
            significant_edges = [
                edge for edge, count in edge_counts.items() 
                if count >= threshold
            ]
            
            # 최종 그래프 생성
            final_graphs = []
            final_graph = nx.DiGraph()
            final_graph.add_edges_from(significant_edges)
            final_graphs.append(final_graph)
            
            logger.info(
                f"Found {len(significant_edges)} significant causal relationships"
            )
            self._print_edge_frequencies(edge_counts, num_samples)
            
            return final_graphs
            
        except Exception as e:
            logger.error(f"Error in causal graph generation: {str(e)}")
            raise

    def _generate_single_graph_with_bic(self, data: pd.DataFrame) -> nx.DiGraph:
        """BIC 스코어를 사용한 단일 그래프 생성"""
        G = nx.DiGraph()
        G.add_nodes_from(data.columns)
        
        returns_cols = ['btc_returns', 'eth_returns']
        onchain_cols = [col for col in data.columns 
                    if col not in returns_cols]
        
        # Forward phase
        for source in onchain_cols:
            for target in returns_cols:
                # BIC 스코어 기반 엣지 추가 여부 결정
                score_with_edge = self._try_add_edge(
                    data, G.copy(), source, target, float('-inf')
                )
                if score_with_edge > float('-inf'):
                    G.add_edge(source, target)
        
        return G

    def _print_edge_frequencies(self, edge_counts: Dict, num_samples: int):
        """엣지 빈도 정보 출력"""
        print("\nEdge Frequencies:")
        for edge, count in edge_counts.items():
            print(f"{edge[0]} -> {edge[1]}: {count}/{num_samples} "
                f"({count/num_samples*100:.1f}%)")

    def _generate_single_graph(self, data: pd.DataFrame, 
                             noise_std: float) -> nx.DiGraph:
        """Generate a single causal graph with noise"""
        noise = np.random.normal(0, noise_std, data.shape)
        noisy_data = data.copy()
        noisy_data.loc[:, :] += noise
        return self._apply_fges(noisy_data)

    def _apply_fges(self, data: pd.DataFrame) -> nx.DiGraph:
        """Apply Fast Greedy Equivalence Search algorithm with improvements"""
        try:
            G = nx.DiGraph()
            G.add_nodes_from(data.columns)
            
            returns_cols = ['btc_returns', 'eth_returns']
            onchain_cols = [col for col in data.columns 
                          if col not in returns_cols and 
                          any(f'{lag}' in col for lag in 
                              map(str, self.config['lag_periods']))]    # 이 부분 수정 필요. f"_lag_" -> f"{lag}" 하든가 그냥 lag 하든가 
            
            # Forward phase with progress tracking
            logger.info("Starting forward phase...")
            G = self._forward_phase(data, G, onchain_cols, returns_cols)
            
            # Backward phase with progress tracking
            logger.info("Starting backward phase...")
            G = self._backward_phase(data, G)
            
            return G
            
        except Exception as e:
            logger.error(f"Error in FGES algorithm: {str(e)}")
            raise

    def _forward_phase(self, data: pd.DataFrame, G: nx.DiGraph, 
                      onchain_cols: List[str], 
                      returns_cols: List[str]) -> nx.DiGraph:
        """Forward phase of FGES algorithm"""
        best_score = self._calculate_bic_score(data, G)
        improved = True
        
        while improved:
            improved = False
            best_new_edge = None
            best_new_score = best_score
            
            # Try adding edges
            for source in onchain_cols:
                for target in returns_cols:
                    if not G.has_edge(source, target):
                        score = self._try_add_edge(data, G, source, 
                                                 target, best_new_score)
                        if score > best_new_score:
                            best_new_score = score
                            best_new_edge = (source, target)
                            improved = True
            
            if improved:
                G.add_edge(*best_new_edge)
                best_score = best_new_score
                logger.info(f"Added edge: {best_new_edge[0]} -> {best_new_edge[1]} "
                          f"(Score: {best_score:.4f})")
                
        return G

    def _backward_phase(self, data: pd.DataFrame, G: nx.DiGraph) -> nx.DiGraph:
        """Backward phase of FGES algorithm"""
        best_score = self._calculate_bic_score(data, G)
        improved = True
        
        while improved:
            improved = False
            best_remove_edge = None
            best_remove_score = best_score
            
            # Try removing edges
            for edge in list(G.edges()):
                score = self._try_remove_edge(data, G, edge, best_remove_score)
                if score > best_remove_score:
                    best_remove_score = score
                    best_remove_edge = edge
                    improved = True
            
            if improved:
                G.remove_edge(*best_remove_edge)
                best_score = best_remove_score
                logger.info(f"Removed edge: {best_remove_edge[0]} -> "
                          f"{best_remove_edge[1]} (Score: {best_score:.4f})")
                
        return G

    def _try_add_edge(self, data: pd.DataFrame, G: nx.DiGraph, 
                      source: str, target: str, 
                      current_best: float) -> float:
        """Try adding an edge and return the score"""
        G_new = G.copy()
        G_new.add_edge(source, target)
        
        if not nx.is_directed_acyclic_graph(G_new):
            return -np.inf
            
        return self._calculate_bic_score(data, G_new)

    def _try_remove_edge(self, data: pd.DataFrame, G: nx.DiGraph, 
                        edge: Tuple[str, str], 
                        current_best: float) -> float:
        """Try removing an edge and return the score"""
        G_new = G.copy()
        G_new.remove_edge(*edge)
        return self._calculate_bic_score(data, G_new)
    

    def calculate_causal_strength(self, graphs: List[nx.DiGraph], 
                            data: pd.DataFrame) -> Dict[str, float]:
        """Calculate causal strength using ATE with improved matching"""
        try:
            logger.info("Calculating causal strengths...")
            causal_strengths = {}
            graph_scores = []
            
            # Calculate BIC scores for all graphs
            for graph in graphs:
                score = self._calculate_bic_score(data, graph)
                graph_scores.append(score)
            
            # Convert scores to weights
            scores_array = np.array(graph_scores)
            max_score = np.max(scores_array)
            weights = np.exp(scores_array - max_score)
            weights = weights / np.sum(weights)
            
            # Process each unique edge
            unique_edges = set([e for g in graphs for e in g.edges()])
            
            for edge in unique_edges:
                source, target = edge
                strengths = []
                edge_weights = []
                
                # Calculate strength for each graph containing the edge
                for graph, weight in zip(graphs, weights):
                    if graph.has_edge(source, target):
                        strength = self._calculate_robust_ate(
                            data=data,
                            source=source,
                            target=target,
                            graph=graph
                        )
                        if strength is not None:
                            strengths.append(strength)
                            edge_weights.append(weight)
                
                if strengths:
                    # Calculate weighted average with normalization
                    edge_weights = np.array(edge_weights)
                    edge_weights = edge_weights / np.sum(edge_weights)
                    causal_strengths[f"{source}->{target}"] = np.average(
                        strengths, 
                        weights=edge_weights
                    )
            
            logger.info(f"Found {len(causal_strengths)} causal relationships")
            return causal_strengths
            
        except Exception as e:
            logger.error(f"Error in causal strength calculation: {str(e)}")
            raise

    def _calculate_robust_ate(self, data: pd.DataFrame, 
                            source: str, target: str, 
                            graph: nx.DiGraph) -> float:
        """Calculate ATE with improved matching and significance testing"""
        try:
            # Get confounders
            confounders = [n for n in graph.nodes() 
                          if n != source and n != target and 
                          (graph.has_edge(n, target) or graph.has_edge(n, source))]
            
            treatment = data[source].values
            outcome = data[target].values
            
            # Calculate propensity scores
            if confounders:
                ps_scores = self._calculate_propensity_scores(
                    data[confounders], treatment)
            else:
                ps_scores = np.ones(len(treatment)) * 0.5
            
            # Match treatment and control groups
            matched_diffs = self._match_and_calculate_effects(
                treatment, outcome, ps_scores)
            
            if not matched_diffs:
                return None
            
            # Calculate ATE
            ate = np.mean(matched_diffs)
            
            # Perform significance test
            if self._is_effect_significant(matched_diffs):
                return ate
            return None
            
        except Exception as e:
            logger.warning(f"Error in ATE calculation: {str(e)}")
            return None

    def _calculate_propensity_scores(self, confounders: pd.DataFrame, 
                                   treatment: np.ndarray) -> np.ndarray:
        """Calculate propensity scores using logistic regression"""
        from sklearn.linear_model import LogisticRegression
        treatment_binary = (treatment > np.median(treatment)).astype(int)
        
        ps_model = LogisticRegression(
            random_state=42, 
            max_iter=1000,
            solver='lbfgs'
        )
        ps_model.fit(confounders, treatment_binary)
        return ps_model.predict_proba(confounders)[:, 1]

    def _match_and_calculate_effects(self, treatment: np.ndarray,
                                   outcome: np.ndarray,
                                   ps_scores: np.ndarray) -> List[float]:
        """Match treatment and control groups using propensity scores"""
        treatment_median = np.median(treatment)
        treated_idx = treatment > treatment_median
        control_idx = ~treated_idx
        
        treated_outcomes = outcome[treated_idx]
        treated_ps = ps_scores[treated_idx]
        control_outcomes = outcome[control_idx]
        control_ps = ps_scores[control_idx]
        
        matched_diffs = []
        
        for t_idx in range(len(treated_outcomes)):
            ps_diff = np.abs(treated_ps[t_idx] - control_ps) # propencity score의 차이가 가장 큰 애들을 추림. 즉 treatment를 받을 확률이 가장 큰 애들만 추린다는 것.
            best_matches = np.argsort(ps_diff)[:3]  # Get top 3 matches # 왜 top 3 만 match 하지? 이러면 ATE가 너무 높아짐.
            
            # Calculate average effect across matches
            effects = [treated_outcomes[t_idx] - control_outcomes[m] 
                      for m in best_matches]
            matched_diffs.append(np.mean(effects))
        
        return matched_diffs

    def _is_effect_significant(self, effects: List[float]) -> bool:
        """Test if treatment effect is statistically significant"""
        _, p_value = stats.ttest_1samp(effects, 0)
        return p_value < self.config['significance_threshold']

    def analyze_returns_causality(self) -> Dict[str, Any]:
        """Analyze causality between onchain metrics and returns"""
        try:
            logger.info("Starting causality analysis...")
            
            # Prepare and normalize data
            # prepare_features에서 normalize한거 다시 normalize_features로 정규화 함  
            # 즉 Normalize 두번 들어감. 수정 필요.
            combined_data = self.prepare_features()
            
            # normalized_data = self.normalize_features(combined_data)
            normalized_data = combined_data
            
            # Generate causal graphs
            causal_graphs = self.generate_causal_graph(normalized_data)
            
            # Calculate causal strengths
            causal_strengths = self.calculate_causal_strength(
                causal_graphs, normalized_data)
            
            # Organize results
            results = self._organize_results(causal_strengths)
            
            # Save results if needed
            self._save_results(results)
            
            # Create visualizations
            self._create_visualizations(
                causal_graphs, causal_strengths, 
                normalized_data, results
            )
            
            logger.info("Causality analysis completed successfully")
            return results
            
        except Exception as e:
            logger.error(f"Error in causality analysis: {str(e)}")
            raise

    def _organize_results(self, causal_strengths: Dict[str, float]) -> Dict[str, Any]:
        """Organize analysis results by cryptocurrency and lag period"""
        results = {
            'btc_factors': {lag: {} for lag in self.config['lag_periods']},
            'eth_factors': {lag: {} for lag in self.config['lag_periods']},
            'causal_strengths': causal_strengths
        }
        
        # Classify results
        for edge, strength in causal_strengths.items():
            source, target = edge.split('->')
            
            for lag in self.config['lag_periods']:
                if f'_lag_{lag}' in source:
                    if 'btc_returns' in target:
                        results['btc_factors'][lag][edge] = strength
                    elif 'eth_returns' in target:
                        results['eth_factors'][lag][edge] = strength
        
        # Add summary statistics
        results['summary'] = self._calculate_summary_statistics(results)
        
        return results

    def _calculate_summary_statistics(self, results: Dict[str, Any]) -> Dict[str, Any]:
        """Calculate summary statistics for the results"""
        summary = {}
        
        for lag in self.config['lag_periods']:
            btc_factors = results['btc_factors'][lag]
            eth_factors = results['eth_factors'][lag]
            
            btc_count = len(btc_factors)
            eth_count = len(eth_factors)
            
            btc_avg = np.mean([abs(v) for v in btc_factors.values()]) \
                     if btc_count > 0 else 0.0
            eth_avg = np.mean([abs(v) for v in eth_factors.values()]) \
                     if eth_count > 0 else 0.0
            
            summary[lag] = {
                'btc_count': btc_count,
                'btc_avg_strength': btc_avg,
                'eth_count': eth_count,
                'eth_avg_strength': eth_avg
            }
            
        return summary

    def _save_results(self, results: Dict[str, Any]) -> None:
        """Save analysis results to file"""
        try:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            filename = f'causal_analysis_results_{timestamp}.json'
            
            with open(filename, 'w') as f:
                json.dump(results, f, indent=4)
                
            logger.info(f"Results saved to {filename}")
            
        except Exception as e:
            logger.warning(f"Failed to save results: {str(e)}")

    def _create_visualizations(self, graphs: List[nx.DiGraph],
                             strengths: Dict[str, float],
                             data: pd.DataFrame,
                             results: Dict[str, Any]) -> None:
        """Create all visualizations"""
        try:
            self.visualize_causal_network(graphs, strengths)
            self.plot_causal_strength_heatmap(results)
            self.plot_temporal_evolution(data, results)
            
        except Exception as e:
            logger.warning(f"Error in visualization creation: {str(e)}")

def main():
    # Configuration settings
    config = {
        'lag_periods': [1, 3, 5, 10, 20],
        'num_samples': 10,
        'noise_std': 0.01,
        'significance_threshold': 0.05,
        'outlier_std': 3,
        'min_causal_strength': 0.1,
        'cache_enabled': True
    }
    
    # Set basic plot style
    plt.rcParams['figure.figsize'] = [12, 8]
    plt.rcParams['axes.grid'] = True
    
    try:
        plt.rcParams['font.family'] = 'Malgun Gothic'
    except:
        logger.warning("Malgun Gothic font not available. Using default font.")
    
    try:
        # Initialize analyzer with config
        logger.info("Initializing analysis...")
        analyzer = CryptoOnchainAnalysis(config)
        
        # Load data
        analyzer.load_data(
            btc_onchain_path="C:/Users/wklee/Desktop/WKLee/FBA/Capstone/fba-capstone/glassnode/glassnode_BTC_tier1.csv",
            eth_onchain_path="C:/Users/wklee/Desktop/WKLee/FBA/Capstone/fba-capstone/glassnode/glassnode_ETH_tier1.csv",
            btc_price_path="C:/Users/wklee/Desktop/WKLee/FBA/Capstone/fba-capstone/binance_1d_BTCUSDT.csv",
            eth_price_path="C:/Users/wklee/Desktop/WKLee/FBA/Capstone/fba-capstone/binance_1d_ETHUSDT.csv"
        )
        
        # Run analysis
        logger.info("Running causality analysis...")
        results = analyzer.analyze_returns_causality()
        
        # Print results for each cryptocurrency and lag period
        print_results(results)
        
    except Exception as e:
        logger.error(f"An error occurred in main execution: {str(e)}")
        traceback.print_exc()

def print_results(results: Dict[str, Any]):
    """Print formatted analysis results"""
    lag_periods = [1, 3, 5, 10, 20]
    
    # Print Bitcoin results
    print("\nBitcoin Returns Causal Factors:")
    print("=" * 40)
    for lag in lag_periods:
        print(f"\nLag {lag} days:")
        btc_factors = results['btc_factors'][lag]
        if btc_factors:
            # Sort factors by absolute strength
            sorted_factors = sorted(
                btc_factors.items(),
                key=lambda x: abs(x[1]),
                reverse=True
            )
            
            # Print top 10 factors
            for factor, strength in sorted_factors[:10]:
                metric = factor.split('->')[0].rsplit('_lag_', 1)[0]
                print(f"{metric:40s}: {strength:>8.4f}")
        else:
            print("No significant causal factors found")
    
    # Print Ethereum results
    print("\nEthereum Returns Causal Factors:")
    print("=" * 40)
    for lag in lag_periods:
        print(f"\nLag {lag} days:")
        eth_factors = results['eth_factors'][lag]
        if eth_factors:
            # Sort factors by absolute strength
            sorted_factors = sorted(
                eth_factors.items(),
                key=lambda x: abs(x[1]),
                reverse=True
            )
            
            # Print top 10 factors
            for factor, strength in sorted_factors[:10]:
                metric = factor.split('->')[0].rsplit('_lag_', 1)[0]
                print(f"{metric:40s}: {strength:>8.4f}")
        else:
            print("No significant causal factors found")
    
    # Print summary statistics
    print("\nSummary Statistics:")
    print("=" * 40)
    for lag in lag_periods:
        print(f"\nLag {lag} days:")
        summary = results['summary'][lag]
        
        if summary['btc_count'] > 0:
            print(f"BTC metrics found: {summary['btc_count']} "
                  f"(Avg. strength: {summary['btc_avg_strength']:.4f})")
            
            # Additional statistics
            btc_factors = results['btc_factors'][lag]
            if btc_factors:
                strengths = np.array(list(btc_factors.values()))
                print(f"  Max strength: {np.max(np.abs(strengths)):.4f}")
                print(f"  Min strength: {np.min(np.abs(strengths)):.4f}")
                print(f"  Std deviation: {np.std(strengths):.4f}")
        else:
            print("No significant BTC metrics found")
        
        if summary['eth_count'] > 0:
            print(f"ETH metrics found: {summary['eth_count']} "
                  f"(Avg. strength: {summary['eth_avg_strength']:.4f})")
            
            # Additional statistics
            eth_factors = results['eth_factors'][lag]
            if eth_factors:
                strengths = np.array(list(eth_factors.values()))
                print(f"  Max strength: {np.max(np.abs(strengths)):.4f}")
                print(f"  Min strength: {np.min(np.abs(strengths)):.4f}")
                print(f"  Std deviation: {np.std(strengths):.4f}")
        else:
            print("No significant ETH metrics found")

if __name__ == "__main__":
    # Configure logging
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler('crypto_analysis.log'),
            logging.StreamHandler()
        ]
    )
    
    # Run main analysis
    main()

2024-11-12 09:12:55,380 - INFO - Initializing analysis...
2024-11-12 09:12:55,497 - INFO - Running causality analysis...
2024-11-12 09:12:55,498 - INFO - Starting causality analysis...
2024-11-12 09:12:55,499 - INFO - Preparing features...
2024-11-12 09:12:55,504 - INFO - Removing outliers...
2024-11-12 09:12:55,549 - INFO - Removing outliers...


Loading data files...
Data loading completed successfully

Data shapes after loading:
BTC metrics shape: (1736, 47)
ETH metrics shape: (1736, 34)
BTC price shape: (1777, 2)
ETH price shape: (1777, 2)
              close   returns
timestamp                    
2020-01-01  7200.85       NaN
2020-01-02  6965.71 -0.033200
2020-01-03  7344.96  0.053015
2020-01-04  7354.11  0.001245
2020-01-05  7358.75  0.000631

Available BTC metrics:
['close', 'active_count', 'block_count', 'block_height', 'block_interval_mean', 'block_interval_median', 'block_size_mean', 'block_size_sum', 'count', 'current', 'difficulty_latest', 'fear_greed', 'gas_limit_tx_median', 'gas_price_median', 'gas_used_median', 'hash_rate_mean', 'marketcap_usd', 'new_non_zero_count', 'price_drawdown_relative', 'price_usd_close', 'purpose_etf_aum_sum', 'purpose_etf_flows_sum', 'purpose_etf_holdings_sum', 'rate', 'receiving_count', 'sending_count', 'sopr', 'staking_deposits_count', 'staking_total_deposits_count', 'staking_total_vol

2024-11-12 09:12:55,647 - INFO - Feature preparation completed successfully
2024-11-12 09:12:55,662 - INFO - Generating 10 causal graphs...
2024-11-12 09:12:55,698 - INFO - Starting forward phase...



Prepared dataset info:
Shape: (997, 407)
Date range: 2021-03-17 to 2024-09-13
Number of features: 407
Memory usage: 3.10345458984375 MB


2024-11-12 09:13:13,816 - INFO - Added edge: eth_sending_count_lag_10 -> eth_returns (Score: -554681.4792)
2024-11-12 09:13:31,361 - INFO - Added edge: btc_gas_used_median_lag_5 -> eth_returns (Score: -554679.0501)
2024-11-12 09:13:47,978 - INFO - Added edge: btc_block_interval_mean_lag_5 -> btc_returns (Score: -554677.1075)
2024-11-12 09:14:05,552 - INFO - Added edge: btc_active_count_lag_20 -> eth_returns (Score: -554675.4709)
2024-11-12 09:14:23,172 - INFO - Added edge: eth_sending_count_lag_10 -> btc_returns (Score: -554673.9846)
2024-11-12 09:14:39,746 - INFO - Added edge: btc_block_count_lag_1 -> eth_returns (Score: -554672.6874)
2024-11-12 09:14:56,323 - INFO - Added edge: btc_block_interval_median_lag_1 -> eth_returns (Score: -554671.4893)
2024-11-12 09:15:12,833 - INFO - Added edge: eth_transfers_volume_mean_lag_10 -> eth_returns (Score: -554670.7781)
2024-11-12 09:15:29,457 - INFO - Added edge: btc_utxo_spent_count_lag_10 -> eth_returns (Score: -554670.2957)
2024-11-12 09:15:


Edge Frequencies:
btc_active_count_lag_20 -> eth_returns: 10/10 (100.0%)
btc_block_count_lag_1 -> eth_returns: 10/10 (100.0%)
btc_block_interval_mean_lag_5 -> btc_returns: 10/10 (100.0%)
btc_block_interval_mean_lag_10 -> eth_returns: 10/10 (100.0%)
btc_fear_greed_lag_1 -> eth_returns: 10/10 (100.0%)
btc_gas_used_median_lag_5 -> eth_returns: 10/10 (100.0%)
btc_utxo_spent_count_lag_10 -> eth_returns: 10/10 (100.0%)
btc_utxo_spent_value_mean_lag_1 -> eth_returns: 10/10 (100.0%)
btc_volume_median_lag_5 -> eth_returns: 10/10 (100.0%)
eth_active_count_lag_10 -> eth_returns: 10/10 (100.0%)
eth_block_interval_mean_lag_5 -> eth_returns: 10/10 (100.0%)
eth_block_size_mean_lag_1 -> eth_returns: 10/10 (100.0%)
eth_marketcap_usd_lag_1 -> eth_returns: 10/10 (100.0%)
eth_price_usd_close_lag_20 -> eth_returns: 10/10 (100.0%)
eth_sending_count_lag_10 -> eth_returns: 10/10 (100.0%)
eth_sending_count_lag_10 -> btc_returns: 10/10 (100.0%)


2024-11-12 09:23:30,415 - INFO - Found 13 causal relationships
2024-11-12 09:23:30,419 - INFO - Results saved to causal_analysis_results_20241112_092330.json
2024-11-12 09:23:30,420 - WARNING - Error in visualization creation: 'CryptoOnchainAnalysis' object has no attribute 'visualize_causal_network'
2024-11-12 09:23:30,421 - INFO - Causality analysis completed successfully



Bitcoin Returns Causal Factors:

Lag 1 days:
eth_sending_count                       :  -0.0064

Lag 3 days:
No significant causal factors found

Lag 5 days:
No significant causal factors found

Lag 10 days:
eth_sending_count                       :  -0.0064

Lag 20 days:
No significant causal factors found

Ethereum Returns Causal Factors:

Lag 1 days:
btc_utxo_spent_value_mean               :  -0.0327
btc_fear_greed                          :   0.0267
eth_active_count                        :  -0.0122
eth_marketcap_usd                       :  -0.0121
btc_utxo_spent_count                    :  -0.0100
eth_sending_count                       :  -0.0083
btc_block_count                         :   0.0060
eth_block_size_mean                     :   0.0051
btc_block_interval_mean                 :  -0.0047

Lag 3 days:
No significant causal factors found

Lag 5 days:
btc_gas_used_median                     :  -0.0093

Lag 10 days:
eth_active_count                        :  -0.0122
btc_ut

In [53]:
# btc_onchain_path="C:/Users/wklee/Desktop/WKLee/FBA/Capstone/fba-capstone/glassnode/glassnode_BTC_tier1.csv"
# eth_onchain_path="C:/Users/wklee/Desktop/WKLee/FBA/Capstone/fba-capstone/glassnode/glassnode_ETH_tier1.csv"
btc_price_path="C:/Users/wklee/Desktop/WKLee/FBA/Capstone/fba-capstone/binance_1d_BTCUSDT.csv"
# eth_price_path="C:/Users/wklee/Desktop/WKLee/FBA/Capstone/fba-capstone/binance_1d_ETHUSDT.csv"

In [ ]:
btc_pri

In [54]:
df = pd.read_csv(btc_price_path)
df

,timestamp,open,high,low,close,volume
0,2020-01-01,7195.24,7255.00,7175.15,7200.85,16792.388165
1,2020-01-02,7200.77,7212.50,6924.74,6965.71,31951.483932
2,2020-01-03,6965.49,7405.00,6871.04,7344.96,68428.500451
3,2020-01-04,7345.00,7404.00,7272.21,7354.11,29987.974977
4,2020-01-05,7354.19,7495.00,7318.00,7358.75,38331.085604
...,...,...,...,...,...,...
1772,2024-11-07,75571.99,76849.99,74416.00,75857.89,44869.422345
1773,2024-11-08,75857.89,77199.99,75555.00,76509.78,36521.099583
1774,2024-11-09,76509.78,76900.00,75714.66,76677.46,16942.079150
1775,2024-11-10,76677.46,81500.00,76492.00,80370.01,61830.100435
